In [2]:
#Home | Zorginstituut Nederland

import time
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd
import hashlib  # Added for generating the content hash
from datetime import datetime  # Added for full timestamps

BASE_URL = "https://www.zorginstituutnederland.nl"
START_URL = "https://www.zorginstituutnederland.nl/werkagenda/kanker"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# Mapping dictionary linking Dutch clinical terms back to clean taxonomy categories
CANCER_MAP = {
    "longkanker": "Longkanker (Lung)",
    "longcarcinoom": "Longkanker (Lung)",
    "borstkanker": "Borstkanker (Breast)",
    "mamma": "Borstkanker (Breast)",
    "prostaatkanker": "Prostaatkanker (Prostate)",
    "darmkanker": "Darmkanker (Colorectal)",
    "colorect": "Darmkanker (Colorectal)",
    "alvleesklier": "Alvleesklierkanker (Pancreatic)",
    "lymfeklierkanker": "Lymfeklierkanker (Lymphoma)",
    "lymfoom": "Lymfeklierkanker (Lymphoma)",
    "bloedkanker": "Bloedkanker (Leukemia)",
    "leukemie": "Bloedkanker (Leukemia)",
    "baarmoeder": "Baarmoederkanker (Uterine/Cervical)",
    "huidkanker": "Huidkanker (Skin)",
    "melanoom": "Huidkanker (Skin)",
    "keelholte": "Hoofd-halskanker (Head & Neck)",
    "strottenhoofd": "Hoofd-halskanker (Head & Neck)"
}

def identify_cancer_type(text_to_search):
    """Parses text blocks to cleanly identify standard target cancer classifications."""
    if not text_to_search:
        return "Algemeen / Multidisciplinair"
        
    normalized = text_to_search.lower()
    found_types = []
    
    for key, standardized_name in CANCER_MAP.items():
        if key in normalized:
            if standardized_name not in found_types:
                found_types.append(standardized_name)
                
    if found_types:
        return " & ".join(found_types)
    return "Algemeen / Onbekend" # Catch-all for systemic issues like general 'molecular diagnostics'

def get_zorginstituut_links():
    """Scrapes the main oncology work agenda page for advisory links."""
    print(f"Connecting to Zorginstituut Hub: {START_URL}")
    response = requests.get(START_URL, headers=HEADERS)
    if response.status_code != 200:
        return {}

    soup = BeautifulSoup(response.text, 'html.parser')
    agendas = {}
    main_area = soup.find('main') or soup.find('div', class_='content') or soup
    
    for link in main_area.find_all('a', href=True):
        href = link['href']
        title = link.get_text(strip=True)
        
        if "/werkagenda/kanker/" in href:
            full_url = BASE_URL + href if href.startswith('/') else href
            if title and "lees meer" not in title.lower():
                if full_url not in agendas.values():
                    agendas[title] = full_url

    print(f"Discovered {len(agendas)} oncology assessment topics to crawl.")
    return agendas

def scrape_assessment_details(title, url):
    """Parses Zorginstituut metadata, descriptions, and maps matching cancer categories."""
    print(f"Scraping advisory content for: [{title}]")
    
    # Generate timestamp down to the second right as processing begins
    scrape_date = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    data_row = {
        "Advisory Topic": title,
        "Standardized Cancer Type": "Algemeen / Onbekend", # Place holder updated below
        "Source URL": url,
        "Scrape Date": scrape_date,  # Added timestamp tracker
        "Content Hash": "N/A",        # Added placeholder for content signature
        "Intro Summary": "N/A",
        "Full Context Block": "N/A"
    }
    
    try:
        response = requests.get(url, headers=HEADERS)
        if response.status_code != 200:
            return data_row

        soup = BeautifulSoup(response.text, 'html.parser')
        
        # 1. Capture Summary Description
        intro_section = soup.find('div', class_='intro') or soup.find('p', class_='lead')
        if intro_section:
            data_row["Intro Summary"] = intro_section.get_text(" ", strip=True)
        else:
            first_p = soup.find('main').find('p') if soup.find('main') else None
            if first_p:
                data_row["Intro Summary"] = first_p.get_text(strip=True)

        # 2. Compile Text Blocks
        main_content = soup.find('main') or soup.find('article') or soup
        body_text_blocks = []
        for element in main_content.find_all(['p', 'li', 'h2', 'h3']):
            text_str = element.get_text(" ", strip=True)
            if text_str and not any(s in text_str.lower() for s in ["deel deze pagina", "downloaden", "printen"]):
                body_text_blocks.append(text_str)
                
        if body_text_blocks:
            full_context = "\n".join(body_text_blocks[:25])
            data_row["Full Context Block"] = full_context
            
            # Generate SHA-256 Content Hash based on the aggregated text data
            data_row["Content Hash"] = hashlib.sha256(full_context.encode('utf-8')).hexdigest()
            
        # 3. --- MAP CANCER TYPE VIA REGEX ENGINE ---
        # Search title and first block to safely categorize the target disease group
        combined_context = f"{title} {data_row['Intro Summary']}"
        data_row["Standardized Cancer Type"] = identify_cancer_type(combined_context)
            
    except Exception as e:
        print(f"Error executing subpage crawl for {url}: {e}")

    return data_row

def main():
    oncology_agenda = get_zorginstituut_links()
    if not oncology_agenda:
        print("Crawl abandoned: No data pointers extracted.")
        return

    scraped_dataset = []
    for title, url in oncology_agenda.items():
        row = scrape_assessment_details(title, url)
        scraped_dataset.append(row)
        time.sleep(1.5)

    df = pd.DataFrame(scraped_dataset)
    
    # Enforcing column ordering consistency explicitly for clarity
    column_order = [
        "Advisory Topic", 
        "Standardized Cancer Type", 
        "Source URL", 
        "Scrape Date", 
        "Content Hash", 
        "Intro Summary", 
        "Full Context Block"
    ]
    df = df[column_order]

    df.to_csv("zorginstituut_categorized_cancer_data.csv", index=False, encoding='utf-8-sig')
    df.to_excel("zorginstituut_categorized_cancer_data.xlsx", index=False)
    print(f"\nCompleted! Generated {len(scraped_dataset)} structured rows successfully.")

if __name__ == "__main__":
    main()

Connecting to Zorginstituut Hub: https://www.zorginstituutnederland.nl/werkagenda/kanker
Discovered 34 oncology assessment topics to crawl.
Scraping advisory content for: [Evaluatie standpunt AFT na een totale borstverwijdering]
Scraping advisory content for: [Evaluatie standpunt Ductoscopie bij pathologische tepeluitvloed]
Scraping advisory content for: [Evaluatie standpunt MammaPrint en Oncotype DX vergoede zorg voor bepaalde groep vrouwen]
Scraping advisory content for: [Evaluatie standpunt Navigatie bronchoscopietechnieken bij verdenking op longkanker]
Scraping advisory content for: [Advies - wel of niet vergoeden acalabrutinib (Calquence®) voor de behandeling van lymfeklierkanker]
Scraping advisory content for: [Advies - wel of niet vergoeden amivantamab (Rybrevant®) in combinatie met lazertinib (Lazcluze®) voor de behandeling van gevorderd niet-kleincellig longkanker]
Scraping advisory content for: [Advies - wel of niet vergoeden belantamab mafodotin (Blenrep®) voor de behandelin